In [ ]:
from pathlib import Path
import runpy

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pixi.toml").exists())
globals().update(runpy.run_path(str(ROOT / "src" / "repo_paths.py")))

import re
import hashlib

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display
import pingouin as pg

from trompy import save_figure_atomic

mpl.rcParams["font.family"] = "Arial"
mpl.rcParams["font.size"] = 10
mpl.rcParams["savefig.transparent"] = True

In [ ]:
# parameter setting

FILE_PATH = DATA_DIR / "food_and_body_weight.xlsx"
OUT_DIR = FIGS_DIR / "panels"
OUT_DIR.mkdir(parents=True, exist_ok=True)

PHASE_ORDER = ["NR1", "PR", "NR2"]
PHASE_MARKERS = {"NR1": "s", "PR": "o", "NR2": "^"}
PHASE_COLOURS = {"NR1": "#EBEBEB", "PR": "#D26959", "NR2": "#8C8C8C"}

COLORS = [PHASE_COLOURS[phase] for phase in PHASE_ORDER]

jitter_x = 0.1

In [ ]:
def read_food_bw_excel(path):
    df = pd.read_excel(path, header=[0, 1])
    df.columns = ["_".join([str(x) for x in col if str(x) != "nan"]).strip("_") for col in df.columns]
    df = df.rename(columns={"mouseid": "MouseID", "mouseid_Unnamed: 0_level_1": "MouseID"})
    df["MouseID"] = df["MouseID"].astype(str).str.strip()

    colmap = {
        "bodyweight_NR1_Start": "NR1_Start",
        "bodyweight_NR1_End": "NR1_End",
        "bodyweight_PR_Start": "PR_Start",
        "bodyweight_PR_End": "PR_End",
        "bodyweight_NR2_Start": "NR2_Start",
        "bodyweight_NR2_End": "NR2_End",
        "food_consumed_NR": "Food_NR1",
        "food_consumed_PR": "Food_PR",
        "food_consumed_NR2": "Food_NR2"
    }

    df = df.rename(columns=colmap)

    df["BWpct_NR1"] = (df["NR1_End"] - df["NR1_Start"]) / df["NR1_Start"] * 100
    df["BWpct_PR"] = (df["PR_End"] - df["PR_Start"]) / df["PR_Start"] * 100
    df["BWpct_NR2"] = (df["NR2_End"] - df["NR2_Start"]) / df["NR2_Start"] * 100

    return df

In [ ]:
# make food intake figure
df = read_food_bw_excel(FILE_PATH)

fig, ax = plt.subplots(figsize=(1.5, 2), gridspec_kw={"left": 0.4, "right": 0.95, "top": 0.95, "bottom": 0.2})

for x, group in enumerate(["Food_NR1", "Food_PR", "Food_NR2"]):
    y = df[group].values.astype(float)
    x_jittered = x + np.random.uniform(-jitter_x, jitter_x, size=len(y))
    ax.scatter(x_jittered, y, label=group, color=COLORS[x], alpha=0.3, edgecolor="k", s=20, clip_on=False, zorder=10, lw=0.75)
    
    ax.plot([x - 0.2, x + 0.2], [np.median(y), np.median(y)], color="black", lw=2.4)
    
for idx, row in df.iterrows():
    x_values = [0, 1, 2]
    y_values = [row["Food_NR1"], row["Food_PR"], row["Food_NR2"]]
    ax.plot(x_values, y_values, color="k", lw=0.8, alpha=0.1)


ax.set_ylabel("Food intake (g)")
ax.set_yticks([30,35,40,45])
ax.set_xticks([0,1,2], labels=["NR1", "PR", "NR2"])

sns.despine(ax=ax, offset=5)

save_figure_atomic(fig, "fig2_food_intake", OUT_DIR)

In [ ]:
# make food intake figure
df = read_food_bw_excel(FILE_PATH)

fig, ax = plt.subplots(figsize=(1.5, 2), gridspec_kw={"left": 0.4, "right": 0.95, "top": 0.95, "bottom": 0.2})

for x, group in enumerate(["BWpct_NR1", "BWpct_PR", "BWpct_NR2"]):
    y = df[group].values.astype(float)
    x_jittered = x + np.random.uniform(-jitter_x, jitter_x, size=len(y))
    ax.scatter(x_jittered, y, label=group, color=COLORS[x], alpha=0.3, edgecolor="k", s=20, clip_on=False, zorder=10, lw=0.75)
    
    ax.plot([x - 0.2, x + 0.2], [np.median(y), np.median(y)], color="black", lw=2.4)
    print(np.median(y))
    
for idx, row in df.iterrows():
    x_values = [0, 1, 2]
    y_values = [row["BWpct_NR1"], row["BWpct_PR"], row["BWpct_NR2"]]
    ax.plot(x_values, y_values, color="k", lw=0.8, alpha=0.1)


ax.set_ylabel("Body weight change (%)")
ax.set_yticks([-5, 0, 5, 10, 15])
ax.set_xticks([0,1,2], labels=["NR1", "PR", "NR2"])
ax.axhline(0, color="dimgray", lw=0.8, ls="--", zorder=0, alpha=0.3)

sns.despine(ax=ax, offset=5)

save_figure_atomic(fig, "fig2_body_weight_change", OUT_DIR)

In [ ]:
bw_df = read_food_bw_excel(FILE_PATH)

bw_long = bw_df.melt(
    id_vars="MouseID",
    value_vars=["BWpct_NR1", "BWpct_PR", "BWpct_NR2"],
    var_name="Phase",
    value_name="BW_change_pct"
)
bw_long["Phase"] = bw_long["Phase"].str.replace("BWpct_", "", regex=False)
bw_long = bw_long.dropna(subset=["BW_change_pct"])
bw_long["Phase"] = pd.Categorical(bw_long["Phase"], categories=PHASE_ORDER, ordered=True)
bw_long = bw_long.sort_values(["MouseID", "Phase"])

bw_desc = bw_long.groupby("Phase", observed=True)["BW_change_pct"].agg(["count", "mean", "median", "std"]).reset_index()

bw_anova_table = pg.rm_anova(
    data=bw_long,
    dv="BW_change_pct",
    within="Phase",
    subject="MouseID",
    detailed=True
).reset_index(drop=True)

phase_posthocs = pg.pairwise_tests(
    data=bw_long,
    dv="BW_change_pct",
    within="Phase",
    subject="MouseID",
    padjust="holm",
    parametric=True,
    return_desc=True
).reset_index(drop=True)

zero_results = []
for phase in PHASE_ORDER:
    values = bw_long.loc[bw_long["Phase"] == phase, "BW_change_pct"].dropna()
    test = pg.ttest(values, 0).iloc[0]
    zero_results.append({
        "phase": phase,
        "n": len(values),
        "mean": values.mean(),
        "T": test["T"],
        "dof": test["dof"],
        "p_uncorrected": test["p_val"],
        "cohen_d": test["cohen_d"]
    })

zero_tests = pd.DataFrame(zero_results)
zero_tests["p_holm"] = pg.multicomp(zero_tests["p_uncorrected"].to_numpy(), method="holm")[1]

print("Body-weight change descriptives")
display(bw_desc.round(3))

print("Repeated-measures ANOVA across phases")
display(bw_anova_table.round(4))

print("Paired posthocs between phases (Holm-corrected)")
display(phase_posthocs[["A", "B", "Paired", "Parametric", "T", "dof", "p_unc", "p_corr", "p_adjust", "hedges"]].round(4))

print("One-sample tests versus zero within each phase (Holm-corrected)")
display(zero_tests.round(4))

In [ ]:
from IPython.display import display
import pingouin as pg

food_df = read_food_bw_excel(FILE_PATH)

food_long = food_df.melt(
    id_vars="MouseID",
    value_vars=["Food_NR1", "Food_PR", "Food_NR2"],
    var_name="Phase",
    value_name="Food_intake_g"
)
food_long["Phase"] = food_long["Phase"].str.replace("Food_", "", regex=False)
food_long = food_long.dropna(subset=["Food_intake_g"])
food_long["Phase"] = pd.Categorical(food_long["Phase"], categories=PHASE_ORDER, ordered=True)
food_long = food_long.sort_values(["MouseID", "Phase"])

food_desc = food_long.groupby("Phase", observed=True)["Food_intake_g"].agg(["count", "mean", "median", "std"]).reset_index()

food_anova_table = pg.rm_anova(
    data=food_long,
    dv="Food_intake_g",
    within="Phase",
    subject="MouseID",
    detailed=True
).reset_index(drop=True)

food_phase_posthocs = pg.pairwise_tests(
    data=food_long,
    dv="Food_intake_g",
    within="Phase",
    subject="MouseID",
    padjust="holm",
    parametric=True,
    return_desc=True
).reset_index(drop=True)

food_zero_results = []
for phase in PHASE_ORDER:
    values = food_long.loc[food_long["Phase"] == phase, "Food_intake_g"].dropna()
    test = pg.ttest(values, 0).iloc[0]
    food_zero_results.append({
        "phase": phase,
        "n": len(values),
        "mean": values.mean(),
        "T": test["T"],
        "dof": test["dof"],
        "p_uncorrected": test["p_val"],
        "cohen_d": test["cohen_d"]
    })

food_zero_tests = pd.DataFrame(food_zero_results)
food_zero_tests["p_holm"] = pg.multicomp(food_zero_tests["p_uncorrected"].to_numpy(), method="holm")[1]

print("Food intake descriptives")
display(food_desc.round(3))

print("Repeated-measures ANOVA across phases")
display(food_anova_table.round(4))

print("Paired posthocs between phases (Holm-corrected)")
display(food_phase_posthocs[["A", "B", "Paired", "Parametric", "T", "dof", "p_unc", "p_corr", "p_adjust", "hedges"]].round(4))

print("One-sample tests versus zero within each phase (Holm-corrected)")
display(food_zero_tests.round(4))